# Qwen-Image 2.1 LoRA 学習ノートブック (完全自動・Colab対応) v1.0

Alibaba の最新基盤画像生成・編集モデル **Qwen-Image-2.1**（DiT 7B + Qwen3-VL 8B + RGBA VAE）の LoRA を学習するためのノートブックです。

**特徴**:
- **操作は最初の3セルのみ**: トークン確認、Google Driveマウント、設定読み込みを済ませたら、あとは最下部まで完全自動で連続実行されます。
- **24GB VRAM対応**: Text Encoder / VAE の事前キャッシングと量子化（int8 / quanto）により、Colab Pro（L4 / A100）等で安定して動作します。
- **ComfyUI即時対応**: AI-ToolkitのネイティブComfyUIプレフィックスで出力されるため、学習完了した `.safetensors` はそのまま ComfyUI で利用可能です。


## 1. あなたの操作 (ここだけ・最初に全部済ませる)


### 1-1. HuggingFaceトークン

Colabの左サイドバーにある **🔑（シークレット）** アイコンから `HF_TOKEN` を登録しておくか、未登録の場合は実行時に入力プロンプトが表示されます。


In [ ]:
import getpass, os

token = None
# 1. Colabのシークレット機能 (🔑) から取得を試みる
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    pass

# 2. 環境変数に既にあるか確認
if not token:
    token = os.environ.get("HF_TOKEN")

# 3. なければ入力プロンプトを表示 (非表示入力)
if not token:
    token = getpass.getpass("HuggingFace Access Token を入力してください (hf_...): ").strip()

if token:
    os.environ["HF_TOKEN"] = token
    print("トークン: OK (設定済み)")
else:
    print("⚠️ トークン未設定! (公開モデルのダウンロードのみ可能)")


### 1-2. Google Driveマウント

ポップアップでGoogleアカウント選択・許可が出たら「許可」をクリックしてください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("DriveマウントOK")


### 1-3. 設定読み込みと確認

`MyDrive/qwen_image21/jobs.json`（または `MyDrive/krea2/jobs.json`）を読み込み、学習予定のジョブ一覧を表示します。
内容を変えたい場合は jobs.json を編集して**このセルだけ再実行**してください。


In [ ]:
import json, os

CFG_PATHS = [
    "/content/drive/MyDrive/qwen_image21/jobs.json",
    "/content/drive/MyDrive/krea2/jobs.json",
]
CFG_PATH = next((p for p in CFG_PATHS if os.path.exists(p)), None)
assert CFG_PATH, f"設定ファイルが見つかりません。MyDrive/qwen_image21/jobs.json を作成してください。\n(検索したパス: {CFG_PATHS})"

cfg = json.load(open(CFG_PATH, encoding="utf-8"))
JOBS = cfg.get("jobs", [])
RESOLUTION = cfg.get("resolution", 1024)
STEPS      = cfg.get("steps")  # 未指定なら calc_steps() が画像枚数から自動決定
LR         = cfg.get("lr", 1e-4)
RANK       = cfg.get("rank", 32)
OUTPUT_DIR = cfg.get("output_dir", "/content/drive/MyDrive/qwen_image21/outputs")

assert JOBS, "JOBSが空です"
for j in JOBS:
    assert j.get("trigger") and j.get("zip"), "JOBSの各要素に trigger と zip が必要です"

steps_disp = STEPS if STEPS else "AUTO(画像枚数ベース)"
print(f"設定ファイル: {CFG_PATH}")
print(f"ジョブ数: {len(JOBS)} / 解像度: {RESOLUTION}px / ステップ: {steps_disp} / lr: {LR} / rank: {RANK}")
print(f"保存先: {OUTPUT_DIR}")
for j in JOBS:
    print(f" - {j['trigger']} <- {j['zip'].rsplit('/', 1)[-1]}")
print("\n問題なければ以降のセルを実行してください（「ランタイム」→「以降のセルを実行」）。")


## 2. 環境構築 (自動・一度実行すればスキップ)

Qwen-Image 2.1 のサポートを含む最新の `ai-toolkit` を導入し、依存関係をセットアップします。


In [ ]:
import os, sys

print("Python:", sys.version.split()[0])
if not os.path.exists("/content/.env_done"):
    if not os.path.exists("/content/ai-toolkit"):
        os.system("git clone -q https://github.com/ostris/ai-toolkit.git /content/ai-toolkit")
        print("ai-toolkit cloned (latest)")
    else:
        os.system("cd /content/ai-toolkit && git pull")
        print("ai-toolkit updated")

    p_req = "/content/ai-toolkit/requirements.txt"
    if os.path.exists(p_req):
        s_req = open(p_req, encoding="utf-8").read()
        if "scipy==1.12.0" in s_req:
            open(p_req, "w", encoding="utf-8").write(s_req.replace("scipy==1.12.0", "scipy>=1.14"))
            print("requirements修正OK (scipy>=1.14)")

    p_base = "/content/ai-toolkit/requirements_base.txt"
    if os.path.exists(p_base):
        s_base = open(p_base, encoding="utf-8").read()
        if "huggingface_hub==" in s_base:
            open(p_base, "w", encoding="utf-8").write(s_base.replace("huggingface_hub==1.23.0", "huggingface_hub>=0.28"))
            print("requirements_base修正OK (huggingface_hub>=0.28)")

    os.system("pip install -q -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
    os.system("pip install -q -r /content/ai-toolkit/requirements.txt")
    # huggingface_hub を含む主要ライブラリをクリーンに最新化
    os.system("pip install -q -U huggingface_hub optimum-quanto diffusers transformers accelerate peft")
    os.system("pip install -q --force-reinstall --no-deps torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126")
    os.system("pip cache purge")
    open("/content/.env_done", "w").write("done")
    print("環境構築完了")
else:
    print("環境構築: 済み (スキップ)")

import torch
props = torch.cuda.get_device_properties(0)
print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {props.total_memory/1e9:.1f}GB")


## 3. 必須パッチ適用 (自動・一度適用すればスキップ)

1. **quanto dequantize フォールバック**: FP8非対応/一部GPUで `qfloat8` がネイティブクラッシュする問題の回避


In [ ]:
import os

if not os.path.exists("/content/.patches_done"):
    try:
        import optimum.quanto
        p1 = os.path.join(os.path.dirname(optimum.quanto.__file__), "tensor", "qtensor_func.py")
        if os.path.exists(p1):
            src = open(p1, encoding="utf-8").read()
            old = '''        elif isinstance(other, QBytesTensor):
            if isinstance(input, QBytesTensor):
                output = torch.ops.quanto.qbytes_mm(input._data, other._data, input._scale * other._scale)
            else:'''
            new = '''        elif isinstance(other, QBytesTensor):
            try:
                if isinstance(input, QBytesTensor):
                    output = torch.ops.quanto.qbytes_mm(input._data, other._data, input._scale * other._scale)
                else:
                    output = torch.ops.quanto.qbytes_mm(input, other._data, other._scale)
            except Exception:
                in_f = input.dequantize() if hasattr(input, "dequantize") else input
                ot_f = other.dequantize() if hasattr(other, "dequantize") else other
                output = torch.matmul(in_f, ot_f)'''
            if old in src:
                open(p1, "w", encoding="utf-8").write(src.replace(old, new))
                print("パッチ1 (quanto dequantize fallback): 適用OK")
            else:
                print("パッチ1: 適用済みまたは該当箇所なし")
    except Exception as e:
        print(f"パッチ1 スキップ: {e}")

    open("/content/.patches_done", "w").write("done")
    print("パッチ適用完了")
else:
    print("パッチ適用: 済み (スキップ)")


## 4. モデルダウンロード (自動・DL済みならスキップ)

Qwen-Image 2.1 のコンポーネントをダウンロードします。
- **Transformer**: `Comfy-Org/Qwen-Image-2.1`（int8 または bf16）
- **Text Encoder**: `text_encoders/qwen3vl_8b_int8_convrot.safetensors`
- **VAE**: `vae/qwen_image_2.1_vae_bf16.safetensors`
- **Configs & Processor**: `Qwen/Qwen-Image-2.1`


In [ ]:
import os, shutil, sys

# モジュールキャッシュ競合対策: メモリ上の古いhuggingface_hubモジュールをリフレッシュ
for mod in list(sys.modules.keys()):
    if mod.startswith("huggingface_hub"):
        del sys.modules[mod]

try:
    from huggingface_hub import hf_hub_download, snapshot_download
except ImportError:
    print("huggingface_hubの不整合を検出。再インストールしてリフレッシュします...")
    os.system("pip install -q --force-reinstall huggingface_hub")
    for mod in list(sys.modules.keys()):
        if mod.startswith("huggingface_hub"):
            del sys.modules[mod]
    from huggingface_hub import hf_hub_download, snapshot_download

BASE_DIR = "/content/models/qwen_image_21"
os.makedirs(f"{BASE_DIR}/diffusion_models", exist_ok=True)
os.makedirs(f"{BASE_DIR}/text_encoders", exist_ok=True)
os.makedirs(f"{BASE_DIR}/vae", exist_ok=True)
os.makedirs("/content/models/qwen_image_21_base", exist_ok=True)

# 1. Configs & Processor (Qwen/Qwen-Image-2.1 から軽量設定ファイルを取得)
CONFIG_CHECK = "/content/models/qwen_image_21_base/model_index.json"
if not os.path.exists(CONFIG_CHECK):
    print("Configs / Tokenizer をダウンロード中 (Qwen/Qwen-Image-2.1)...")
    snapshot_download(
        "Qwen/Qwen-Image-2.1",
        allow_patterns=["*.json", "*.txt", "tokenizer*", "text_encoder/*.json", "vae/*.json", "transformer/*.json", "scheduler/*.json"],
        local_dir="/content/models/qwen_image_21_base",
    )
    print("Configs DL完了")

# 2. VAE (0.68 GB)
VAE_PATH = f"{BASE_DIR}/vae/qwen_image_2.1_vae_bf16.safetensors"
if not os.path.exists(VAE_PATH):
    print("VAE をダウンロード中...")
    p = hf_hub_download("Comfy-Org/Qwen-Image-2.1", "vae/qwen_image_2.1_vae_bf16.safetensors")
    shutil.copy(p, VAE_PATH)
    print(f"VAE DL完了: {os.path.getsize(VAE_PATH)/1e9:.2f} GB")

# 3. Text Encoder (int8: 9.35 GB)
TE_PATH = f"{BASE_DIR}/text_encoders/qwen3vl_8b_int8_convrot.safetensors"
if not os.path.exists(TE_PATH):
    print("Text Encoder (int8) をダウンロード中...")
    p = hf_hub_download("Comfy-Org/Qwen-Image-2.1", "text_encoders/qwen3vl_8b_int8_convrot.safetensors")
    shutil.copy(p, TE_PATH)
    print(f"Text Encoder DL完了: {os.path.getsize(TE_PATH)/1e9:.2f} GB")

# 4. Transformer (int8: 7.26 GB)
TRANS_PATH = f"{BASE_DIR}/diffusion_models/qwen_image_2.1_int8_convrot.safetensors"
if not os.path.exists(TRANS_PATH):
    print("Transformer (int8) をダウンロード中...")
    p = hf_hub_download("Comfy-Org/Qwen-Image-2.1", "diffusion_models/qwen_image_2.1_int8_convrot.safetensors")
    shutil.copy(p, TRANS_PATH)
    print(f"Transformer DL完了: {os.path.getsize(TRANS_PATH)/1e9:.2f} GB")

# ディスク節約: HF重複キャッシュ削除
if os.path.isdir("/root/.cache/huggingface/hub"):
    os.system("rm -rf /root/.cache/huggingface/hub")
    print("HF一時キャッシュを削除してディスクを解放しました")

print("\n全モデル準備完了: MODELS_DONE")


## 5. 関数定義 (自動・定義するだけ)

データセット前処理、YAML設定生成、学習実行、LoRA保存の各関数を定義します。


In [ ]:
import glob, os, shutil, zipfile

META_TAGS = {"character name", "character", "name", "symbols", "text", "english text",
             "dated", "score", "no humans", "commentary", "translated", "artist name"}

def calc_steps(n_imgs):
    # 画像枚数から学習ステップ数を自動決定
    if n_imgs <= 10:
        return 400
    if n_imgs <= 30:
        return 1000
    if n_imgs <= 50:
        return 1000
    if n_imgs <= 100:
        return 1250
    if n_imgs <= 150:
        return 1500
    return 1800

def split_tags_and_prose(caption):
    parts = [p.strip() for p in caption.split(",") if p.strip()]
    prose_idx = None
    for i, p in enumerate(parts):
        words = p.split()
        if len(words) >= 4 and p[0].isupper() and not p.isupper() and not p[0].isdigit():
            prose_idx = i
            break
    if prose_idx is None:
        return parts, []
    return parts[:prose_idx], parts[prose_idx:]

def fix_caption(caption, trigger):
    tags, prose = split_tags_and_prose(caption)
    if trigger and trigger in tags:
        tags = [t for t in tags if t != trigger]
    tags = [t for t in tags if t.lower() not in META_TAGS]
    keep = []
    for t in tags:
        tlow = t.lower()
        if len(t.split()) == 1 and any(tlow in o.lower().split() and o.lower() != tlow for o in tags):
            continue
        keep.append(t)
    seen, deduped = set(), []
    for t in keep:
        if t.lower() not in seen:
            seen.add(t.lower())
            deduped.append(t)
    res = [trigger] + prose + deduped if trigger else prose + deduped
    return ", ".join(res)

def prepare_dataset(zip_path, trigger):
    if not os.path.exists(zip_path) and not zip_path.startswith("/"):
        cand = os.path.join("/content/drive", zip_path)
        if os.path.exists(cand):
            zip_path = cand
    assert os.path.exists(zip_path), f"ZIPが見つかりません: {zip_path}"

    os.system("rm -rf /content/dataset")
    os.makedirs("/content/dataset", exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall("/content/dataset")

    # サブフォルダに展開されたファイルを直下に移動
    for f in glob.glob("/content/dataset/**/*", recursive=True):
        if os.path.isfile(f) and os.path.dirname(f) != "/content/dataset":
            shutil.move(f, os.path.join("/content/dataset", os.path.basename(f)))

    # 大文字小文字を含む全画像拡張子を取得
    valid_exts = {".png", ".jpg", ".jpeg", ".webp"}
    imgs = sorted([
        os.path.join("/content/dataset", f) for f in os.listdir("/content/dataset")
        if os.path.isfile(os.path.join("/content/dataset", f)) and os.path.splitext(f)[1].lower() in valid_exts
    ])
    caps = sorted(glob.glob("/content/dataset/*.txt"))

    assert len(imgs) >= 1, f"ZIP内に有効な画像 (png/jpg/webp) が見つかりません: {zip_path}"

    changed = 0
    for txt in caps:
        raw = open(txt, encoding="utf-8", errors="ignore").read().strip()
        fixed = fix_caption(raw, trigger)
        if fixed != raw:
            changed += 1
        with open(txt, "w", encoding="utf-8") as f:
            f.write(fixed + "\n")

    print(f"データセット準備OK: 画像{len(imgs)}枚 / キャプション{len(caps)}個 (修正{changed}個) / trigger='{trigger}'")
    if caps:
        print("例:", open(caps[0], encoding="utf-8").read().strip()[:120], "...")
    return len(imgs)


In [ ]:
import datetime, glob, json, os, re, shutil, torch

# ---------- config生成 ----------
def make_config(trigger, res, steps, lr, rank):
    if isinstance(res, (int, float)):
        res_list = f"[512, 768, {int(res)}]"
    elif isinstance(res, list):
        res_list = str(res)
    else:
        res_list = "[512, 768, 1024]"

    yaml = f"""---
job: extension
config:
  name: "{trigger}_colab"
  process:
    - type: 'sd_trainer'
      training_folder: "output/{trigger}_colab"
      device: cuda:0
      network:
        type: "lora"
        linear: {rank}
        linear_alpha: {rank}
      save:
        dtype: float16
        save_every: 100
        max_step_saves_to_keep: 15
      datasets:
        - folder_path: "/content/dataset"
          caption_ext: "txt"
          caption_dropout_rate: 0.05
          shuffle_tokens: false
          cache_latents_to_disk: true
          resolution: {res_list}
      train:
        batch_size: 1
        cache_text_embeddings: true
        steps: {steps}
        gradient_accumulation: 1
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: "flowmatch"
        optimizer: "adamw8bit"
        lr: {lr}
        dtype: bf16
        timestep_type: linear
      model:
        name_or_path: "Comfy-Org/Qwen-Image-2.1"
        extras_name_or_path: "Qwen/Qwen-Image-2.1"
        arch: "qwen_image_2"
        quantize: true
        qtype_te: "qfloat8"
        quantize_te: true
        low_vram: true
      sample:
        sampler: "flowmatch"
        sample_every: 250
        width: 1024
        height: 1024
        prompts:
          - "{trigger}, 1girl, solo, A high quality illustration in the style of {trigger}"
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 3
        sample_steps: 25
meta:
  name: "[name]"
"""
    cfg_file = "/content/ai-toolkit/qwen21_colab.yaml"
    with open(cfg_file, "w", encoding="utf-8") as f:
        f.write(yaml)
    print(f"config書込OK: {cfg_file} / resolution {res_list} / {steps} steps / lr {lr} / rank {rank}")

# ---------- 学習 ----------
def train(trigger):
    os.chdir("/content/ai-toolkit")
    ret = os.system("python -u run.py qwen21_colab.yaml")
    if ret != 0:
        raise RuntimeError("学習スクリプト (run.py) がエラー終了しました。直上の出力ログを確認してください。")
    outs = sorted(glob.glob(f"output/{trigger}_colab/{trigger}_colab/*.safetensors"))
    print(f"学習完了: 出力ファイル数 {len(outs)}個")
    assert outs, "学習出力 (.safetensors) が見つかりません。"

# ---------- 納品 (Drive保存) ----------
def deliver(trigger, steps, lr, sweep_min=None):
    out_dir = OUTPUT_DIR
    os.makedirs(out_dir, exist_ok=True)
    jst = datetime.timezone(datetime.timedelta(hours=9))
    dstr = datetime.datetime.now(jst).strftime("%Y%m%d")

    folder = f"/content/ai-toolkit/output/{trigger}_colab/{trigger}_colab"
    ckpts = sorted(glob.glob(os.path.join(folder, f"{trigger}_colab_*.safetensors")))
    final = os.path.join(folder, f"{trigger}_colab.safetensors")
    if os.path.exists(final):
        ckpts.append(final)

    delivered = []
    for ckpt in ckpts:
        base = os.path.basename(ckpt)
        m = re.search(r"_(\\d{9})\\.safetensors$", base)
        if m:
            step = int(m.group(1))
            if sweep_min is not None and step < sweep_min:
                continue
            label = f"{step}steps"
        else:
            label = f"{steps}steps_final"

        dst_name = f"{trigger}_qwen21_{dstr}_{label}_lr{lr}.safetensors"
        dst = os.path.join(out_dir, dst_name)
        shutil.copy(ckpt, dst)
        delivered.append(dst_name)
        print(f"Drive保存: {dst_name} ({os.path.getsize(ckpt)/1e6:.1f} MB)")

    print(f"納品完了: 計 {len(delivered)} 個のファイルを {out_dir} に保存しました。")
    return delivered

print("関数定義OK (prepare_dataset / make_config / train / deliver)")


## 6. 全ジョブ実行 (自動・本丸)

JOBSの全データセットを順番に: **展開 → タグ修正 → 学習 → Drive保存** します。
- 1ジョブがエラーしても**残りは続行**し、最後に `FAILED: [...]` を表示します。
- **再実行時は完了済みジョブを自動スキップ**します。


In [ ]:
import os, traceback, torch

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
gpu = torch.cuda.get_device_name(0)
print(f"GPU: {gpu} / VRAM: {vram_gb:.1f}GB")

res = RESOLUTION
if vram_gb < 20:
    res = 768
    print("VRAM 20GB未満: 最大解像度を 768px に調整")

DONE = "/content/jobs_done"
os.makedirs(DONE, exist_ok=True)

N = len(JOBS)
failed = []

for i, job in enumerate(JOBS, 1):
    trig, zip_ = job["trigger"], job["zip"]
    marker = os.path.join(DONE, f"{trig}.done")
    if os.path.exists(marker):
        print(f"\nSKIP: {trig} は完了済みです (スキップ)")
        continue

    print(f"\n{'='*64}\nJOB {i}/{N}: {trig} ({zip_.rsplit('/', 1)[-1]})\n{'='*64}")
    os.system(f"rm -rf /content/ai-toolkit/output/{trig}_colab")

    try:
        n_imgs = prepare_dataset(zip_, trig)
        steps = job.get("steps") or STEPS or calc_steps(n_imgs)
        print(f"→ ステップ数: {steps} (画像 {n_imgs} 枚)")

        make_config(trig, res, steps, LR, RANK)
        train(trig)
        deliver(trig, steps, LR, sweep_min=job.get("sweep_min"))

        with open(marker, "w") as f:
            f.write("done")
        print(f"JOB_DONE: {trig} ({i}/{N})")
    except Exception as e:
        print(f"❌ エラー発生: {trig}")
        traceback.print_exc()
        failed.append(trig)

print("\n" + "="*64)
if failed:
    print(f"⚠️ 失敗したジョブがあります: {failed}")
else:
    print("🎉 全ジョブが正常に完了しました！")
print("="*64)


## 完了! 次のステップ

1. 各LoRAは Google Drive の保存先（デフォルト: `MyDrive/qwen_image21/outputs/`）に保存されています。
2. **ComfyUIでの利用**:
   - `ComfyUI/models/loras/` に配置して `LoraLoader` で読み込みます。
   - Qwen-Image 2.1 の DiT モデルおよび Qwen3-VL テキストエンコーダのワークフローに接続して生成できます。
